# Workbench

Um notebook para o dia-a-dia: carrega a exportação **uma única vez** (a cache do `00_setup.ipynb` garante-o) e corre qualquer análise sobre a execução que escolher, ou sobre várias.

In [2]:
%run 00_setup.ipynb   # funções partilhadas e configuração da suite

In [3]:
RUN_ID = REFERENCE_RUN   # = 2, a execucao de referencia (a adjudicada); mude para analisar outra
ann    = load("annotations.csv", run=RUN_ID)
gold   = load("gold.csv", run=RUN_ID)
status = load("status.csv", run=RUN_ID)
fail   = load("failures.csv", run=RUN_ID)
dist   = load("pattern_distribution.csv", run=RUN_ID)
sample = load("sample.csv", run=RUN_ID)
meta   = load("meta.csv", run=RUN_ID).iloc[0].to_dict()

print(f"run {RUN_ID}: {meta['label']}  |  painel: {meta['panel']}")
print(f"{len(ann)} linhas de anotacao, {len(gold)} linhas de gold, {len(status)} linhas de estado, {len(fail)} linhas de falha")

run 2: llm_panel #2  |  painel: deepseek/deepseek-v4-flash;meta-llama/llama-3.3-70b-instruct;openai/gpt-oss-120b;qwen/qwen3-next-80b-a3b-instruct
67317 linhas de anotacao, 5000 linhas de gold, 3600 linhas de estado, 57 linhas de falha


## Escolher a execução
A omissão é a execução de referência (`REFERENCE_RUN`, a adjudicada). Para analisar outra, mude o `RUN` abaixo e re-corra as células seguintes; para comparar várias, use a última secção. Nada é recarregado do disco: a cache trata disso.

In [4]:
RUN = REFERENCE_RUN
ann_r  = load("annotations.csv", run=RUN)
gold_r = load("gold.csv", run=RUN)
print(f"run {RUN}: {len(ann_r)} anotacoes")

run 2: 67317 anotacoes


## As três análises sobre a execução escolhida

In [5]:
agreement_table(ann_r).sort_values("pos_votes", ascending=False).head(8)

,code,alpha,kappa,ac1,p_pos,p_neg,pos_votes
6,PM-1,0.730,0.732,0.963,0.750,0.982,237
15,TM-1,0.670,0.672,0.956,0.690,0.979,231
2,PE-1,0.245,0.248,0.966,0.262,0.983,79
3,PE-2,0.117,0.078,0.968,0.130,0.984,63
1,DR-2,0.381,0.401,0.977,0.391,0.989,63
8,PM-3,0.634,0.621,0.990,0.632,0.995,47
14,SE-3,0.433,0.443,0.987,0.450,0.994,38
7,PM-2,0.259,0.359,0.988,0.262,0.994,29


In [6]:
cell_df = cells(ann_r, gold_r, "open")
macro(calibration(cell_df))

,precision,recall,f1
model,,,
deepseek/deepseek-v4-flash,0.518,0.139,0.204
meta-llama/llama-3.3-70b-instruct,0.512,0.197,0.272
openai/gpt-oss-120b,0.477,0.122,0.179
panel-majority,0.564,0.126,0.194
qwen/qwen3-next-80b-a3b-instruct,0.656,0.359,0.446


In [7]:
bootstrap_macro_f1(cell_df)

{'deepseek/deepseek-v4-flash': (np.float64(0.148), np.float64(0.245)),
 'meta-llama/llama-3.3-70b-instruct': (np.float64(0.204), np.float64(0.314)),
 'openai/gpt-oss-120b': (np.float64(0.12), np.float64(0.226)),
 'qwen/qwen3-next-80b-a3b-instruct': (np.float64(0.36), np.float64(0.488)),
 'panel-majority': (np.float64(0.132), np.float64(0.242))}

In [8]:
adjudication_direction(ann_r, gold_r, "open")

adicoes independentes (painel ausente, autor presente): 234 / 4335 celulas


dir,confirmation,replacement,override_rate
code,,,
PE-2,192,37,0.162
TM-1,191,37,0.162
PM-1,201,27,0.118
PE-3,205,23,0.101
PE-1,210,18,0.079
SE-3,211,17,0.075
DR-2,211,17,0.075
PM-2,215,13,0.057
PM-4,216,12,0.053


## Saúde dos dados da execução escolhida

In [9]:
status_r = load("status.csv", run=RUN)
print(status_by_model(status_r).to_string())
print("adjudicadas em painel parcial: %d / %d (%.1f%%)" % adjudicated_on_partial(status_r, gold_r, "open"))

                                   parse_err  total   pct
modelSlug                                                
meta-llama/llama-3.3-70b-instruct         47    900  5.22
openai/gpt-oss-120b                        8    900  0.89
deepseek/deepseek-v4-flash                 1    900  0.11
qwen/qwen3-next-80b-a3b-instruct           1    900  0.11
adjudicadas em painel parcial: 16 / 231 (6.9%)


## Comparar várias execuções
Qualquer métrica, em ciclo, sobre a lista que quiser (o zip tem de as conter: exporte com `?run=all`).

In [10]:
RUNS_TO_COMPARE = [1, 2, 3, 4, 8, 9]
rows = {}
for r in RUNS_TO_COMPARE:
    a, g = load("annotations.csv", run=r), load("gold.csv", run=r)
    m = macro(calibration(cells(a, g, "open")))
    rows[f"run{r}"] = {"painel_f1": m.loc["panel-majority", "f1"], "melhor_f1": m.drop(index="panel-majority")["f1"].max()}
pd.DataFrame(rows).T

,painel_f1,melhor_f1
run1,0.380,0.694
run2,0.194,0.446
run3,0.198,0.345
run4,0.147,0.297
run8,0.114,0.400
run9,0.319,0.379
